In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("dirty_cafe_sales.csv")
df.head()

print("Shape:", df.shape)
print()
df.info()
print()
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

for col in ["Item", "Quantity", "Payment Method", "Location"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

# Save a copy of the original so we can compare before vs after later
df_original = df.copy()

# Replace ERROR and UNKNOWN with real missing values
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

print(df.isnull().sum())

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

df.info()

# Each item has one fixed price, find it using the most common price per item
price_map = df.groupby("Item")["Price Per Unit"].agg(lambda x: x.mode()[0]).to_dict()
print(price_map)

# 1. Fill missing prices using the item's menu price
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))

# 2. Fill missing items from the price, but only where the price belongs to one item
#    (Cake/Juice are both 3.0 and Sandwich/Smoothie are both 4.0, so those stay unknown)
price_counts = pd.Series(price_map).value_counts()
unique_prices = {p: item for item, p in price_map.items() if price_counts[p] == 1}
df["Item"] = df["Item"].fillna(df["Price Per Unit"].map(unique_prices))

# 3. Use the formula Total Spent = Quantity x Price Per Unit
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
df["Quantity"] = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Total Spent"] / df["Quantity"])

# 4. Label what we can't recover
df["Item"] = df["Item"].fillna("Unknown")
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")

# 5. Drop rows with no date, since a date can't be calculated
df = df.dropna(subset=["Transaction Date"])

print(df.isnull().sum())
print("Rows left:", len(df))

df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))

price_counts = pd.Series(price_map).value_counts()
unique_prices = {p: item for item, p in price_map.items() if price_counts[p] == 1}
df["Item"] = df["Item"].fillna(df["Price Per Unit"].map(unique_prices))

df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
df["Quantity"] = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Total Spent"] / df["Quantity"])

df["Item"] = df["Item"].fillna("Unknown")
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")

df = df.dropna(subset=["Transaction Date"])

print(df.isnull().sum())
print("Rows left:", len(df))


Shape: (10000, 8)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB

Nulls per column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate rows: 0
Item
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie   

In [5]:
# Drop rows where we can't recover the numbers
df = df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])

print(df.isnull().sum())
print("Rows left:", len(df))

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64
Rows left: 9514


In [6]:
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(col, "| range:", df[col].min(), "to", df[col].max(), "| outliers:", len(outliers))

Quantity | range: 1.0 to 5.0 | outliers: 0
Price Per Unit | range: 1.0 to 5.0 | outliers: 0
Total Spent | range: 1.0 to 25.0 | outliers: 259


In [7]:
# Clean up spaces and capitalisation in text columns
for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].str.strip().str.title()

print(df["Item"].unique())
print(df["Payment Method"].unique())
print(df["Location"].unique())

# Check that Total Spent = Quantity x Price Per Unit
mismatch = df[(df["Quantity"] * df["Price Per Unit"]).round(2) != df["Total Spent"].round(2)]
print("Rows where the maths doesn't match:", len(mismatch))

['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'Unknown' 'Sandwich' 'Tea'
 'Juice']
['Credit Card' 'Cash' 'Unknown' 'Digital Wallet']
['Takeaway' 'In-Store' 'Unknown']
Rows where the maths doesn't match: 0


In [8]:
before_nulls = df_original.replace(["ERROR", "UNKNOWN"], np.nan).isnull().sum().sum()

summary = pd.DataFrame({
    "Before": [len(df_original), before_nulls, df_original.duplicated().sum(), "4 of 8"],
    "After": [len(df), df.isnull().sum().sum(), df.duplicated().sum(), "8 of 8"]
}, index=["Row count", "Missing values", "Duplicate rows", "Columns with correct data type"])

print(summary)

df.to_csv("cleaned_cafe_sales.csv", index=False)
print("Saved cleaned_cafe_sales.csv")

                                Before   After
Row count                        10000    9514
Missing values                   10082       0
Duplicate rows                       0       0
Columns with correct data type  4 of 8  8 of 8
Saved cleaned_cafe_sales.csv


## Data Cleaning Summary

**Data quality issues found:** The dataset had 10,082 missing or invalid values. Besides empty cells, the columns contained "ERROR" and "UNKNOWN" entries, and every column was stored as text.

**Missing data strategy:**
- ERROR/UNKNOWN entries were converted to real missing values so they could be handled consistently.
- Price Per Unit was filled from each item's fixed menu price.
- Quantity, Price and Total Spent were recovered using Total Spent = Quantity × Price Per Unit.
- Payment Method and Location were missing in about 30-40% of rows, so they were labelled "Unknown" instead of deleting those rows.
- 460 rows with no Transaction Date were removed, because a date can't be calculated.
- 26 rows where two of the three numeric values were missing were removed.

**Duplicates:** 0 duplicate rows were found, so none were removed.

**Standardisation:** Spaces and capitalisation in text columns were made consistent. Numeric columns were converted to numbers and the date column to datetime.

**Outliers:** The IQR method flagged 259 high Total Spent values, but they are valid orders (maximum 5 × 5.00 = 25), so they were kept.

**Result:** 10,000 rows reduced to 9,514 clean rows, with 0 missing values. The cleaned data was saved as cleaned_cafe_sales.csv.